# 06 — ARIMA Model

This notebook implements a deliberately simple regional ARIMA benchmark under the frozen
Phase 3 protocol. Candidate configuration selection uses 2021 validation only; 2022 is
evaluated once after the order is fixed. No large automatic order search is performed.


## 1. Load and verify frozen inputs


In [ ]:
from pathlib import Path
import importlib.util
import json
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.dpi": 120, "savefig.dpi": 300})

def locate_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "phase4_utils.py").exists():
            return candidate
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = locate_root(Path.cwd())
spec = importlib.util.spec_from_file_location("phase4_utils", PROJECT_ROOT / "src" / "phase4_utils.py")
u = importlib.util.module_from_spec(spec)
spec.loader.exec_module(u)
evaluation = u.load_evaluation_module(PROJECT_ROOT)
splits = u.load_splits(PROJECT_ROOT)
print({name: data["combined"].shape for name, data in splits.items()})


In [ ]:
for split_name, data in splits.items():
    assert data["X"][u.KEYS].equals(data["y"][u.KEYS])
    assert sorted(data["X"]["year"].unique().tolist()) == u.SPLIT_YEARS[split_name]
    assert data["X"]["region"].nunique() == 13
    assert not data["X"].duplicated(u.KEYS).any()
    assert list(data["X"].columns) == u.KEYS + u.PREDICTORS
print("Frozen protocol and target alignment verified.")


## 2. Statistical feasibility and reconstructed histories

Each region has only two training target years (2019–2020). The frozen lag columns provide the
same validated 2017–2018 observations, allowing a four-point training history without accessing
or rebuilding earlier pipelines. Even four points are inadequate for reliable parameter
inference; therefore only random-walk ARIMA(0,1,0) with and without drift is considered.
Stationarity tests are not meaningful at this sample size and are not reported as evidence.


In [ ]:
try:
    from statsmodels.tsa.arima.model import ARIMA
    import statsmodels
except ImportError as exc:
    raise ImportError("Notebook 06 requires statsmodels. Install it before execution.") from exc

MODEL_DIR = PROJECT_ROOT / "models" / "arima"
RESULT_DIR = PROJECT_ROOT / "results" / "arima"
FIGURE_DIR = PROJECT_ROOT / "figures" / "arima"
for directory in (MODEL_DIR, RESULT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

u.write_json(RESULT_DIR / "environment.json", u.package_versions(
    ["numpy", "pandas", "statsmodels", "matplotlib", "seaborn"]
))

def training_history(region):
    rows = splits["train"]["combined"].query("region == @region").sort_values("year")
    first = rows.iloc[0]
    values = [first["consumption_lag_2"], first["consumption_lag_1"], *rows[u.TARGET].tolist()]
    return pd.Series(values, index=[2017, 2018, 2019, 2020], dtype=float)

histories = {region: training_history(region) for region in sorted(splits["train"]["combined"]["region"].unique())}
assert all(len(series) == 4 for series in histories.values())


## 3. Validation-only configuration selection

`no_drift` is the standard random walk. `linear_drift` estimates a deterministic drift in the
differenced series. Convergence warnings and fit failures are retained in diagnostics; a
failed regional fit falls back to the last observation for that candidate and is flagged.


In [ ]:
CANDIDATES = {
    "ARIMA(0,1,0)_no_drift": {"order": (0, 1, 0), "trend": "n"},
    "ARIMA(0,1,0)_linear_drift": {"order": (0, 1, 0), "trend": "t"},
}

def fit_forecast(series, configuration):
    model = ARIMA(series.astype(float), order=configuration["order"], trend=configuration["trend"])
    fitted = model.fit()
    return fitted, float(fitted.forecast(1).iloc[0])

candidate_predictions, diagnostics = [], []
validation = splits["validation"]["combined"].sort_values(u.KEYS)
for candidate, configuration in CANDIDATES.items():
    for row in validation.itertuples():
        series = histories[row.region]
        try:
            fitted, forecast = fit_forecast(series, configuration)
            status, message = "success", ""
            aic = float(fitted.aic)
        except Exception as exc:
            forecast = float(series.iloc[-1])
            status, message, aic = "fallback_last_observation", f"{type(exc).__name__}: {exc}", np.nan
        candidate_predictions.append({"candidate": candidate, "year": row.year, "region": row.region,
                                      "actual": getattr(row, u.TARGET), "predicted": forecast})
        diagnostics.append({"stage": "validation", "candidate": candidate, "region": row.region,
                            "status": status, "message": message, "aic": aic, "n_history": len(series)})

candidate_predictions = pd.DataFrame(candidate_predictions)
candidate_metrics = []
for candidate, group in candidate_predictions.groupby("candidate"):
    metrics = evaluation.evaluate_regression(group["actual"], group["predicted"],
                                             model_name=candidate, split="validation")
    candidate_metrics.append(metrics)
candidate_metrics = pd.DataFrame(candidate_metrics).sort_values(["RMSE", "model"]).reset_index(drop=True)
selected_name = candidate_metrics.iloc[0]["model"]
selected_config = CANDIDATES[selected_name]
display(candidate_metrics)
print("Selected using validation only:", selected_name)


## 4. Fixed-configuration validation and test forecasts


In [ ]:
validation_selected = candidate_predictions.query("candidate == @selected_name")
validation_output = u.prediction_frame(
    validation_selected[u.KEYS], validation_selected["actual"], validation_selected["predicted"],
    selected_name, "validation"
)

test = splits["test"]["combined"].sort_values(u.KEYS)
test_rows, final_models = [], {}
for row in test.itertuples():
    validation_value = splits["validation"]["combined"].loc[
        splits["validation"]["combined"]["region"].eq(row.region), u.TARGET
    ].iloc[0]
    series = pd.concat([histories[row.region], pd.Series([validation_value], index=[2021])])
    try:
        fitted, forecast = fit_forecast(series, selected_config)
        final_models[row.region] = fitted
        status, message, aic = "success", "", float(fitted.aic)
    except Exception as exc:
        forecast = float(series.iloc[-1])
        status, message, aic = "fallback_last_observation", f"{type(exc).__name__}: {exc}", np.nan
    test_rows.append({"year": row.year, "region": row.region, "actual": getattr(row, u.TARGET),
                      "predicted": forecast})
    diagnostics.append({"stage": "test", "candidate": selected_name, "region": row.region,
                        "status": status, "message": message, "aic": aic, "n_history": len(series)})
test_raw = pd.DataFrame(test_rows)
test_output = u.prediction_frame(test_raw[u.KEYS], test_raw["actual"], test_raw["predicted"],
                                 selected_name, "test")
predictions = pd.concat([validation_output, test_output], ignore_index=True)
metrics = pd.DataFrame([u.evaluate_prediction_frame(evaluation, validation_output),
                        u.evaluate_prediction_frame(evaluation, test_output)])
predictions.to_csv(RESULT_DIR / "predictions.csv", index=False, float_format="%.15g")
metrics.to_csv(RESULT_DIR / "metrics.csv", index=False, float_format="%.15g")
candidate_metrics.to_csv(RESULT_DIR / "candidate_validation_metrics.csv", index=False)
pd.DataFrame(diagnostics).to_csv(RESULT_DIR / "diagnostics.csv", index=False)
with (MODEL_DIR / "regional_arima_models.pkl").open("wb") as handle:
    pickle.dump(final_models, handle)
u.write_json(MODEL_DIR / "configuration.json", {
    "selected_name": selected_name, "order": list(selected_config["order"]),
    "trend": selected_config["trend"], "selection_split": "validation",
    "forecast_horizon_years": 1, "fallback_rule": "last observation on fit failure"
})
display(metrics)


## 5. Diagnostics and figures


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, split_name in zip(axes, ["validation", "test"]):
    group = predictions.query("dataset_split == @split_name").sort_values("actual")
    ax.plot(group["actual"].to_numpy()/1e9, label="Observed", marker="o")
    ax.plot(group["predicted"].to_numpy()/1e9, label="ARIMA", marker="s")
    ax.set_title(split_name.title()); ax.set_ylabel("Billion kWh"); ax.set_xlabel("Regions ordered by actual")
axes[1].legend(frameon=False)
fig.tight_layout(); fig.savefig(FIGURE_DIR / "predictions.png", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=predictions, x=predictions["residual"]/1e9, hue="dataset_split",
             kde=True, element="step", ax=ax)
ax.set_xlabel("Residual (billion kWh)"); ax.set_title("ARIMA Residual Distribution", weight="bold")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "residuals.png", bbox_inches="tight"); plt.show()

regional = predictions.groupby("region", as_index=False)["absolute_error"].mean().sort_values("absolute_error")
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=regional, x=regional["absolute_error"]/1e9, y="region", color="#1F4E78", ax=ax)
ax.set_xlabel("Mean absolute error (billion kWh)"); ax.set_ylabel("")
ax.set_title("ARIMA Regional Performance", weight="bold")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "regional_performance.png", bbox_inches="tight"); plt.show()


## 6. Summary


In [ ]:
failures = pd.DataFrame(diagnostics).query("status != 'success'")
report = f'''# ARIMA Summary Report

- Selected configuration: **{selected_name}**
- Selection data: 2021 validation only
- Test year: 2022, evaluated once after configuration selection
- Regional histories: four observations for validation and five for test
- Fit/fallback events: {len(failures)}

## Scientific limitations

The series are far too short for reliable stationarity testing, parameter inference, or complex
ARIMA identification. AIC values are diagnostic only and were not used for cross-configuration
selection. Regional fits may be numerically fragile. All failures are retained in diagnostics,
and the documented last-observation fallback prevents silent row loss.
'''
(RESULT_DIR / "summary_report.md").write_text(report, encoding="utf-8")
print("Notebook 06 complete.")
